# Cartographie prédictive GWS — RandomForestRegressor

Notebook Google Colab généré à partir du script Python d'origine : **`RFR.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Le calcul est lancé par la cellule `main()` en fin de notebook.
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_RF_IDW_APIIDW_TWI_DISTRIVER` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS
#  (TRAIN = longue période, TEST = % des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  - Modèle RandomForestRegressor sur grille GLDAS (~25 km)
#
#  - Variables explicatives :
#       * Precipitations_IDW         → CHIRPS-IDW (journalier, issu IDW)
#       * Precipitations_API_IDW     → API-IDW (journalier, issu IDW)
#       * MODIS_LST_Clipped          → LST MODIS (journalier)
#       * MODIS_NDVI_Clipped         → NDVI MODIS (16 jours, étendu)
#       * MODIS_ET_Clipped           → ET MODIS (~8 jours, étendu)
#       * ERA5_Qsb_Clipped           → Qsb ERA5 (journalier)
#       * SMAP_RZSM_Clipped          → RZSM SMAP (journalier)
#       * SMAP_SoilM_Clipped         → SoilM SMAP (journalier)
#       * TWI (statique)
#       * Distance_To_River (statique)
#
#  - Cible :
#       * GLDAS_GWS_Clipped          → GWS (journalier)
#
#  - Features finales :
#       [P_IDW, API_IDW, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, TWI, DistRiver, sinDOY, cosDOY]
#
#  ➤ Périodes :
#    TRAIN pool : 2021-01-01 → 2025-06-30
#    TEST = TEST_FRACTION des dates (aléatoire dans la fenêtre TRAIN)
#    FORECAST : 2025-07-01 → 2025-10-30
#
#  ✅ + FEATURE SELECTION "MIN" :
#     - Permutation Importance sur X_val
#     - On garde importance_mean > 0 (et au moins FS_MIN_KEEP)
#     - Refit RF + mapping/forecast avec les features retenues
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab (scikit-learn, numpy et joblib le sont).

In [ ]:
!pip install -q rasterio

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date, timedelta

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 0. Régularisation RandomForest (ANTI-OVERFITTING)

In [ ]:
# ------------------------------------------------------------
# 0) Régularisation RandomForest (ANTI-OVERFITTING)
# ------------------------------------------------------------
# MODE = "AUTO"  -> teste plusieurs configs régularisées et choisit celle qui généralise le mieux
# MODE = "FIXED" -> applique une config régularisée fixe (RF_FIXED_REG_PARAMS)
REG_MODE = "AUTO"   # "AUTO" ou "FIXED"

# Base commune
RF_BASE_PARAMS = {
    "n_estimators": 800,
    "bootstrap": True,   # requis pour OOB
}

# Config régularisée fixe (si REG_MODE="FIXED")
RF_FIXED_REG_PARAMS = {
    "max_depth": 25,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "max_features": 0.4,
    # "max_samples": 0.8,  # activé seulement si sklearn le supporte
    # "ccp_alpha": 0.0,    # optionnel si dispo
}

# Petites variantes régularisées à tester (si REG_MODE="AUTO")
RF_REG_CANDIDATES = [
    {"max_depth": 30, "min_samples_split": 20, "min_samples_leaf": 10, "max_features": 0.5, "max_samples": 0.9},
    {"max_depth": 25, "min_samples_split": 20, "min_samples_leaf": 10, "max_features": 0.4, "max_samples": 0.8},
    {"max_depth": 25, "min_samples_split": 30, "min_samples_leaf": 15, "max_features": 0.4, "max_samples": 0.8},
    {"max_depth": 20, "min_samples_split": 30, "min_samples_leaf": 20, "max_features": 0.35, "max_samples": 0.7},
    {"max_depth": 18, "min_samples_split": 40, "min_samples_leaf": 25, "max_features": 0.30, "max_samples": 0.7},
]

# Pénalité anti-overfitting
OVERFIT_GAP_WEIGHT = 0.60

### 0.1 FEATURE SELECTION "MIN"

In [ ]:
# ------------------------------------------------------------
# 0.1) FEATURE SELECTION "MIN"
# ------------------------------------------------------------
DO_FEATURE_SELECTION = True

# permutation importance
FS_N_REPEATS = 5
FS_RANDOM_STATE = 42

# règle de garde
FS_MIN_KEEP = 6  # au moins 6 features, même si importance<=0

FEATURE_NAMES = [
    "P_IDW",
    "API_IDW",
    "LST",
    "NDVI_ext",
    "ET_ext",
    "Qsb",
    "RZSM",
    "SoilM",
    "TWI",
    "DistRiver",
    "sinDOY",
    "cosDOY",
]


def _supported_params_only(param_dict):
    """Filtre les clés non supportées par la version sklearn installée (ex: max_samples si ancienne version)."""
    probe = RandomForestRegressor()
    allowed = set(probe.get_params().keys())
    return {k: v for k, v in param_dict.items() if k in allowed}


def _fit_and_score_rf(X_tr, y_tr, X_val, y_val, params, seed=42):
    """Entraîne + calcule un score qui favorise la généralisation (et pénalise l'overfit)."""
    full_params = {}
    full_params.update(RF_BASE_PARAMS)
    full_params.update(params)
    full_params = _supported_params_only(full_params)

    model = RandomForestRegressor(
        **full_params,
        n_jobs=-1,
        random_state=seed,
        oob_score=True,
    )

    t0 = time.time()
    model.fit(X_tr, y_tr)
    t1 = time.time()

    ytr_pred = model.predict(X_tr)
    yval_pred = model.predict(X_val)

    r2_tr = r2_score(y_tr, ytr_pred)
    r2_val = r2_score(y_val, yval_pred)

    gap = max(0.0, r2_tr - r2_val)
    oob_r2 = getattr(model, "oob_score_", np.nan)

    sel_score = r2_val - OVERFIT_GAP_WEIGHT * gap

    info = {
        "model": model,
        "fit_ms": (t1 - t0) * 1000.0,
        "r2_tr": r2_tr,
        "r2_val": r2_val,
        "gap": gap,
        "oob_r2": oob_r2,
        "sel_score": sel_score,
        "params_used": full_params,
    }
    return info


def choose_regularized_params(X_tr, y_tr, X_val, y_val):
    """Choisit automatiquement une config régularisée qui généralise bien."""
    print("\n=== SÉLECTION AUTO DES PARAMÈTRES RF RÉGULARISÉS (ANTI-OVERFITTING) ===")

    best = None
    for i, cand in enumerate(RF_REG_CANDIDATES, start=1):
        info = _fit_and_score_rf(X_tr, y_tr, X_val, y_val, cand, seed=42)

        print(f"\n--- Candidate #{i} ---")
        for k in sorted(cand.keys()):
            if k in info["params_used"]:
                print(f"  {k} = {info['params_used'][k]}")
        print(
            f"  ➤ fit = {info['fit_ms']:.2f} ms | "
            f"R²_tr={info['r2_tr']:.4f} | R²_val={info['r2_val']:.4f} | "
            f"gap={info['gap']:.4f} | OOB={info['oob_r2']:.4f}"
        )
        print(f"  ➤ sel_score = {info['sel_score']:.4f}")

        if (best is None) or (info["sel_score"] > best["sel_score"]):
            best = info

    print("\n✅ Meilleure config régularisée retenue :")
    for k in sorted(best["params_used"].keys()):
        if k in RF_BASE_PARAMS or k in {"max_depth", "min_samples_split", "min_samples_leaf",
                                        "max_features", "max_samples", "ccp_alpha"}:
            print(f"  - {k} = {best['params_used'][k]}")
    print(
        f"   (R²_val={best['r2_val']:.4f}, gap={best['gap']:.4f}, "
        f"OOB={best['oob_r2']:.4f}, sel_score={best['sel_score']:.4f})"
    )

    return best["params_used"]


def select_features_minimal_perm(rf_model, X_val, y_val, feature_names):
    """
    Sélection MIN :
      - calcule permutation importance sur X_val
      - garde importances_mean > 0
      - sécurité : garde au moins FS_MIN_KEEP meilleures
    """
    print("\n=== FEATURE SELECTION (Permutation Importance sur X_val) ===")

    perm = permutation_importance(
        rf_model,
        X_val,
        y_val,
        n_repeats=FS_N_REPEATS,
        random_state=FS_RANDOM_STATE,
        n_jobs=-1,
        scoring="r2",
    )

    importances = perm.importances_mean.astype("float32")
    stds = perm.importances_std.astype("float32")

    order = np.argsort(importances)[::-1]

    print("\nImportances (mean ± std) triées :")
    for idx in order:
        print(f"  - {feature_names[idx]:>10s} : {importances[idx]: .6f} ± {stds[idx]: .6f}")

    keep_idx = np.where(importances > 0)[0]

    # sécurité : au moins FS_MIN_KEEP
    min_keep = min(FS_MIN_KEEP, len(feature_names))
    if keep_idx.size < min_keep:
        keep_idx = order[:min_keep]

    keep_idx = np.array(sorted(keep_idx.tolist()), dtype=int)
    kept_names = [feature_names[i] for i in keep_idx]

    print("\n✅ Features retenues :", kept_names)
    print("   Nb retenues :", len(keep_idx), "/", len(feature_names))

    return keep_idx

## 1. Paramètres généraux

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1) Paramètres généraux
# ------------------------------------------------------------
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

PRECIP_IDW_DIR = os.path.join(BASE_DIR, "Precipitations_IDW")
API_IDW_DIR    = os.path.join(BASE_DIR, "Precipitations_API_IDW")

LST_DIR    = os.path.join(BASE_DIR, "MODIS_LST_Clipped")
NDVI_DIR   = os.path.join(BASE_DIR, "MODIS_NDVI_Clipped")
ET_DIR     = os.path.join(BASE_DIR, "MODIS_ET_Clipped")
GWS_DIR    = os.path.join(BASE_DIR, "GLDAS_GWS_Clipped")
QSB_DIR    = os.path.join(BASE_DIR, "ERA5_Qsb_Clipped")
RZSM_DIR   = os.path.join(BASE_DIR, "SMAP_RZSM_Clipped")
SOILM_DIR  = os.path.join(BASE_DIR, "SMAP_SoilM_Clipped")

TWI_PATH = os.path.join(BASE_DIR, "TWI", "TWI.tif")
DIST_RIVER_PATH = os.path.join(BASE_DIR, "Distance_To_River", "Distance_To_River.tif")

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_RF_IDW_APIIDW_TWI_DISTRIVER")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE = -9999.0
TEST_FRACTION = 0.10  # 10%

TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = date(2025, 7, 1)
FORECAST_END   = date(2025, 10, 30)

print("=== PARAMÈTRES GÉNÉRAUX ===")
print(" BASE_DIR :", BASE_DIR)
print(" NAME_FILE :", NAME_FILE)
print(" PRECIP_IDW_DIR :", PRECIP_IDW_DIR)
print(" API_IDW_DIR    :", API_IDW_DIR)
print(" TWI_PATH :", TWI_PATH)
print(" DIST_RIVER_PATH :", DIST_RIVER_PATH)
print(" OUT_DIR :", OUT_DIR)
print(" TRAIN_START/END :", TRAIN_START, "→", TRAIN_END)
print(" TEST_FRACTION :", TEST_FRACTION)
print(" FORECAST_START/END :", FORECAST_START, "→", FORECAST_END)
print(" REG_MODE :", REG_MODE)
print(" DO_FEATURE_SELECTION :", DO_FEATURE_SELECTION)
print("============================\n")

if not os.path.isfile(TWI_PATH):
    raise FileNotFoundError(f"❌ Raster TWI introuvable : {TWI_PATH}")
if not os.path.isfile(DIST_RIVER_PATH):
    raise FileNotFoundError(f"❌ Raster Distance_To_River introuvable : {DIST_RIVER_PATH}")

## 2. DATES EXCLUES (LST manquante 2022-10-11 → 2022-10-22)

In [ ]:
# ------------------------------------------------------------
# 2) DATES EXCLUES (LST manquante 2022-10-11 → 2022-10-22)
# ------------------------------------------------------------
EXCLUDED_DATES = set(date(2022, 10, d) for d in range(11, 23))
print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)

## 3. Fonctions utilitaires

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 3) Fonctions utilitaires
# ------------------------------------------------------------
def date_range(d0: date, d1: date):
    d = d0
    while d <= d1:
        yield d
        d += timedelta(days=1)


def extract_date_from_filename(path):
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire la date depuis : {name}")
    return datetime.strptime(m.group(1), "%Y%m%d").date()


def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path):
    sections = {}
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections


def build_date_to_path_dict_from_section(sections, section_name):
    if section_name not in sections:
        raise RuntimeError(f"Section '{section_name}' absente de {NAME_FILE}")
    d = {}
    for rel_path in sections[section_name]:
        abs_path = os.path.normpath(os.path.join(BASE_DIR, rel_path))
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            date_key = extract_date_from_filename(abs_path)
            d[date_key] = abs_path
        except ValueError as e:
            print("⚠️", e)
    return d


def read_raster_raw(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata


def to_float_with_nan(arr, nodata):
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr


def reproject_to_template(
    src_arr,
    src_profile,
    template_profile,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average,
):
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32",
    )
    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata
    reproject(**kwargs)
    return dst_arr


def fill_nan_with_mean(arr):
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)


def build_prev_date_support(date_dict):
    dates_sorted = sorted(date_dict.keys())

    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]

    return get_prev


def write_geotiff(out_path, arr2d, template_profile, nodata_value=NODATA_VALUE):
    prof = template_profile.copy()
    prof.update(
        dtype="float32",
        count=1,
        nodata=nodata_value,
        compress="deflate",
        predictor=2,
    )
    out = arr2d.astype("float32")
    out = np.where(np.isfinite(out), out, nodata_value)
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(out, 1)


# --- Masque "Regions" reconstruit à partir des rasters ---
REGION_MASK = None


def update_region_mask(precip_res, lst_res, ndvi_res, template_profile):
    global REGION_MASK
    base_mask = ~(np.isnan(precip_res) & np.isnan(lst_res) & np.isnan(ndvi_res))
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit à partir des rasters (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask


def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé, on prend toute la grille.")
        REGION_MASK = np.ones((template_profile["height"], template_profile["width"]), dtype=bool)
    return REGION_MASK

### 3.1 Rasters statiques sur la grille template (cache)

In [ ]:
# ------------------------------------------------------------
# 3.1) Rasters statiques sur la grille template (cache)
# ------------------------------------------------------------
TWI_ON_TEMPLATE = None
DIST_ON_TEMPLATE = None


def ensure_twi_on_template(template_profile):
    global TWI_ON_TEMPLATE
    if TWI_ON_TEMPLATE is not None:
        return TWI_ON_TEMPLATE
    twi_raw, twi_prof, twi_nodata = read_raster_raw(TWI_PATH)
    twi_res = reproject_to_template(
        twi_raw, twi_prof, template_profile,
        src_nodata=twi_nodata, dst_nodata=np.nan,
        resampling=Resampling.bilinear
    )
    twi_f = fill_nan_with_mean(twi_res).astype("float32")
    TWI_ON_TEMPLATE = twi_f
    print("\n✅ TWI chargé et reprojeté sur la grille GLDAS (template).")
    print("   - min/max (approx):", float(np.nanmin(twi_f)), "/", float(np.nanmax(twi_f)))
    return TWI_ON_TEMPLATE


def ensure_dist_on_template(template_profile):
    global DIST_ON_TEMPLATE
    if DIST_ON_TEMPLATE is not None:
        return DIST_ON_TEMPLATE
    dist_raw, dist_prof, dist_nodata = read_raster_raw(DIST_RIVER_PATH)
    dist_res = reproject_to_template(
        dist_raw, dist_prof, template_profile,
        src_nodata=dist_nodata, dst_nodata=np.nan,
        resampling=Resampling.bilinear
    )
    dist_f = fill_nan_with_mean(dist_res).astype("float32")
    DIST_ON_TEMPLATE = dist_f
    print("\n✅ Distance_To_River chargé et reprojeté sur la grille GLDAS (template).")
    print("   - min/max (approx):", float(np.nanmin(dist_f)), "/", float(np.nanmax(dist_f)))
    return DIST_ON_TEMPLATE

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## Pipeline principal — définition de `main()`

Cette cellule ne fait que définir la fonction ; le calcul est lancé dans la cellule suivante.

In [ ]:
def main():
    # ------------------------------------------------------------
    # 4) Indexation des rasters par date
    # ------------------------------------------------------------
    print("\n=== INDEXATION DES FICHIERS PAR DATE (via name_files.txt) ===")
    sections = parse_name_file(NAME_FILE)

    precip_dict = build_date_to_path_dict_from_section(sections, "Precipitations_IDW")
    api_dict    = build_date_to_path_dict_from_section(sections, "Precipitations_API_IDW")

    lst_dict    = build_date_to_path_dict_from_section(sections, "MODIS_LST_Clipped")
    ndvi_dict   = build_date_to_path_dict_from_section(sections, "MODIS_NDVI_Clipped")
    et_dict     = build_date_to_path_dict_from_section(sections, "MODIS_ET_Clipped")
    gws_dict    = build_date_to_path_dict_from_section(sections, "GLDAS_GWS_Clipped")
    qsb_dict    = build_date_to_path_dict_from_section(sections, "ERA5_Qsb_Clipped")
    rzsm_dict   = build_date_to_path_dict_from_section(sections, "SMAP_RZSM_Clipped")
    soilm_dict  = build_date_to_path_dict_from_section(sections, "SMAP_SoilM_Clipped")

    print("Nb dates Precip_IDW :", len(precip_dict))
    print("Nb dates API_IDW    :", len(api_dict))
    print("Nb dates LST        :", len(lst_dict))
    print("Nb dates NDVI       :", len(ndvi_dict))
    print("Nb dates ET         :", len(et_dict))
    print("Nb dates GWS        :", len(gws_dict))
    print("Nb dates Qsb        :", len(qsb_dict))
    print("Nb dates RZSM       :", len(rzsm_dict))
    print("Nb dates SoilM      :", len(soilm_dict))

    if len(precip_dict) == 0:
        raise RuntimeError("⚠️ precip_dict est vide. Vérifie [Precipitations_IDW] dans name_files.txt.")
    if len(api_dict) == 0:
        raise RuntimeError("⚠️ api_dict est vide. Vérifie [Precipitations_API_IDW] dans name_files.txt.")

    get_ndvi_date_for = build_prev_date_support(ndvi_dict)
    get_et_date_for   = build_prev_date_support(et_dict)

    # ------------------------------------------------------------
    # 5) Dates TRAIN / TEST
    # ------------------------------------------------------------
    print(f"\n=== CONSTRUCTION DES DATES TRAIN / TEST ({int(TEST_FRACTION*100)}% des dates pour TEST) ===")

    all_common_dates = sorted(
        set(precip_dict.keys())
        & set(api_dict.keys())
        & set(lst_dict.keys())
        & set(gws_dict.keys())
        & set(qsb_dict.keys())
        & set(rzsm_dict.keys())
        & set(soilm_dict.keys())
    )
    print("Nombre total de dates communes :", len(all_common_dates))
    if not all_common_dates:
        raise RuntimeError("⚠️ Aucune date commune entre les variables journalières (incluant IDW).")

    date_candidates = [
        d for d in all_common_dates
        if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
    ]
    if len(date_candidates) == 0:
        raise RuntimeError("⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusion).")

    train_dates, test_dates = train_test_split(
        date_candidates,
        test_size=TEST_FRACTION,
        random_state=42,
        shuffle=True,
    )
    train_dates = sorted(train_dates)
    test_dates  = sorted(test_dates)

    print(f"\nFenêtre TRAIN : {TRAIN_START} → {TRAIN_END}")
    print(f" → Dates train : {len(train_dates)}")
    print(f"\nTEST = {int(TEST_FRACTION*100)}%")
    print(f" → Dates test  : {len(test_dates)}")

    # ------------------------------------------------------------
    # 6) Build dataset X/y
    # ------------------------------------------------------------
    print("\n=== CONSTRUCTION DU DATASET (toutes les dates TRAIN + TEST) ===")

    X_train_list, y_train_list = [], []
    X_test_list,  y_test_list  = [], []

    template_profile = None

    def build_Xy_for_date(date_key, template_profile, role="train"):
        ndvi_date = get_ndvi_date_for(date_key)
        if ndvi_date is None:
            print(f"⏭️ [{role}] {date_key} : aucun NDVI <= date, ignoré.")
            return None, None, template_profile

        et_date = get_et_date_for(date_key)
        if et_date is None:
            print(f"⏭️ [{role}] {date_key} : aucun ET <= date, ignoré.")
            return None, None, template_profile

        if date_key not in gws_dict:
            print(f"⏭️ [{role}] {date_key} : GWS manquant, ignoré.")
            return None, None, template_profile

        gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
        gws_arr = to_float_with_nan(gws_raw, gws_nodata)

        if template_profile is None:
            template_profile = gws_prof
            print("\nGrille GLDAS (template) :")
            print(" - CRS :", template_profile["crs"])
            print(" - Taille:", template_profile["width"], "x", template_profile["height"])
            print(" - Résolution approx (deg) :", template_profile["transform"][0])

            ensure_twi_on_template(template_profile)
            ensure_dist_on_template(template_profile)

        # Vérif journaliers
        if (
            date_key not in precip_dict
            or date_key not in api_dict
            or date_key not in lst_dict
            or date_key not in qsb_dict
            or date_key not in rzsm_dict
            or date_key not in soilm_dict
        ):
            print(f"⏭️ [{role}] {date_key} : variable journalière manquante, ignoré.")
            return None, None, template_profile

        precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
        api_raw,    api_prof,    api_nodata    = read_raster_raw(api_dict[date_key])
        lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
        ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
        et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
        qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
        rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
        soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

        precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
        api_res    = reproject_to_template(api_raw,    api_prof,    template_profile, src_nodata=api_nodata,    dst_nodata=np.nan)
        lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata,    dst_nodata=np.nan)
        ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata,   dst_nodata=np.nan)
        et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata,     dst_nodata=np.nan)
        qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata,    dst_nodata=np.nan)
        rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata,   dst_nodata=np.nan)
        soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata,  dst_nodata=np.nan)

        update_region_mask(precip_res, lst_res, ndvi_res, template_profile)
        region_mask = get_region_mask(template_profile)

        precip_f = fill_nan_with_mean(precip_res)
        api_f    = fill_nan_with_mean(api_res)
        lst_f    = fill_nan_with_mean(lst_res)
        ndvi_f   = fill_nan_with_mean(ndvi_res)
        et_f     = fill_nan_with_mean(et_res)
        qsb_f    = fill_nan_with_mean(qsb_res)
        rzsm_f   = fill_nan_with_mean(rzsm_res)
        soilm_f  = fill_nan_with_mean(soilm_res)

        twi_2d  = ensure_twi_on_template(template_profile)
        dist_2d = ensure_dist_on_template(template_profile)

        mask_valid = (~np.isnan(gws_arr)) & region_mask
        if not np.any(mask_valid):
            print(f"⏭️ [{role}] {date_key} : aucun pixel valide, ignoré.")
            return None, None, template_profile

        doy = date_key.timetuple().tm_yday
        sin_doy = np.sin(2 * np.pi * doy / 365.0)
        cos_doy = np.cos(2 * np.pi * doy / 365.0)

        n_pixels = np.count_nonzero(mask_valid)
        sin_plane = np.full(n_pixels, sin_doy, dtype="float32")
        cos_plane = np.full(n_pixels, cos_doy, dtype="float32")

        X_day = np.stack(
            [
                precip_f[mask_valid],
                api_f[mask_valid],
                lst_f[mask_valid],
                ndvi_f[mask_valid],
                et_f[mask_valid],
                qsb_f[mask_valid],
                rzsm_f[mask_valid],
                soilm_f[mask_valid],
                twi_2d[mask_valid],
                dist_2d[mask_valid],
                sin_plane,
                cos_plane,
            ],
            axis=1,
        )
        y_day = gws_arr[mask_valid]

        row_mask = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
        if not np.any(row_mask):
            print(f"⏭️ [{role}] {date_key} : pixels invalides (NaN features), ignoré.")
            return None, None, template_profile

        X_day = X_day[row_mask]
        y_day = y_day[row_mask]

        print(f" ✔️ [{role}] {date_key} : {X_day.shape[0]} pixels utilisés.")
        return X_day, y_day, template_profile

    for d in train_dates:
        X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="train")
        if X_day is None:
            continue
        X_train_list.append(X_day)
        y_train_list.append(y_day)

    for d in test_dates:
        X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="test")
        if X_day is None:
            continue
        X_test_list.append(X_day)
        y_test_list.append(y_day)

    if len(X_train_list) == 0:
        raise RuntimeError("⚠️ Aucun pixel d'entraînement valide n'a été trouvé.")
    if len(X_test_list) == 0:
        raise RuntimeError("⚠️ Aucun pixel de test valide n'a été trouvé.")

    X_train = np.vstack(X_train_list).astype("float32")
    y_train = np.concatenate(y_train_list).astype("float32")
    X_test  = np.vstack(X_test_list).astype("float32")
    y_test  = np.concatenate(y_test_list).astype("float32")

    print(f"\nPixels TRAIN : {X_train.shape[0]}")
    print(f"Pixels TEST  : {X_test.shape[0]}")
    print("✅ Nombre de features :", X_train.shape[1])

    # ------------------------------------------------------------
    # 7) Modèle RandomForestRegressor + OOB + Régularisation
    # ------------------------------------------------------------
    print("\n=== ENTRAÎNEMENT DU MODÈLE RANDOM FOREST (RÉGULARISÉ ANTI-OVERFITTING) ===")

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, shuffle=True
    )
    print(f"Taille X_tr : {X_tr.shape}, X_val : {X_val.shape}")

    # Choix des hyperparamètres régularisés
    if REG_MODE.upper() == "AUTO":
        rf_params_used = choose_regularized_params(X_tr, y_tr, X_val, y_val)
    else:
        fixed = {}
        fixed.update(RF_BASE_PARAMS)
        fixed.update(RF_FIXED_REG_PARAMS)
        rf_params_used = _supported_params_only(fixed)

        print("\n=== PARAMÈTRES RF RÉGULARISÉS (FIXED) ===")
        for k in sorted(rf_params_used.keys()):
            print(f"  - {k} = {rf_params_used[k]}")

    # Modèle (fit sur X_tr) : on le fit une première fois (pour metrics + feature selection)
    rf_model = RandomForestRegressor(
        **rf_params_used,
        n_jobs=-1,
        random_state=42,
        oob_score=True,
    )

    print("\n🔄 Entraînement Random Forest (sur X_tr) + calcul OOB...")
    t0 = time.time()
    rf_model.fit(X_tr, y_tr)
    t1 = time.time()
    training_time_ms = (t1 - t0) * 1000.0
    print(f" ➤ Fit initial terminé en {training_time_ms:.2f} ms")

    # ------------------------------------------------------------
    # 7.A) FEATURE SELECTION (MIN) : permutation importance sur X_val
    # ------------------------------------------------------------
    selected_idx = np.arange(X_tr.shape[1], dtype=int)  # par défaut: toutes
    selected_feature_names = FEATURE_NAMES.copy()

    if DO_FEATURE_SELECTION:
        selected_idx = select_features_minimal_perm(rf_model, X_val, y_val, FEATURE_NAMES)
        selected_feature_names = [FEATURE_NAMES[i] for i in selected_idx]

        # Slice matrices
        X_tr    = X_tr[:, selected_idx]
        X_val   = X_val[:, selected_idx]
        X_test  = X_test[:, selected_idx]
        X_train = X_train[:, selected_idx]

        print("\n✅ Après feature selection :")
        print("  - Nb features retenues :", X_tr.shape[1])
        print("  - Features :", selected_feature_names)

        # Refit du modèle après sélection
        rf_model = RandomForestRegressor(
            **rf_params_used,
            n_jobs=-1,
            random_state=42,
            oob_score=True,
        )

        print("\n🔁 Refit RF après feature selection (sur X_tr)...")
        t0 = time.time()
        rf_model.fit(X_tr, y_tr)
        t1 = time.time()
        print(f" ➤ Refit terminé en {(t1 - t0)*1000.0:.2f} ms")

    # ------------------------------------------------------------
    # 7.B) Métriques train/val/test (après sélection si activée)
    # ------------------------------------------------------------
    y_tr_pred   = rf_model.predict(X_tr)
    y_val_pred  = rf_model.predict(X_val)
    y_test_pred = rf_model.predict(X_test)

    r2_tr   = r2_score(y_tr, y_tr_pred)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
    mae_tr  = mean_absolute_error(y_tr, y_tr_pred)

    r2_val   = r2_score(y_val, y_val_pred)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
    mae_val  = mean_absolute_error(y_val, y_val_pred)

    r2_test   = r2_score(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae_test  = mean_absolute_error(y_test, y_test_pred)

    # OOB métriques (sur X_tr uniquement)
    oob_pred = getattr(rf_model, "oob_prediction_", None)
    oob_r2 = getattr(rf_model, "oob_score_", np.nan)
    oob_mae = np.nan
    oob_rmse = np.nan
    if oob_pred is not None:
        mask_oob = np.isfinite(oob_pred) & np.isfinite(y_tr)
        if np.any(mask_oob):
            oob_mae = mean_absolute_error(y_tr[mask_oob], oob_pred[mask_oob])
            oob_rmse = np.sqrt(mean_squared_error(y_tr[mask_oob], oob_pred[mask_oob]))

    print("\n╔═════════ MODÈLE RANDOM FOREST (RÉGULARISÉ) ═════════╗")
    print(f" ➤ Nb features utilisées : {X_tr.shape[1]}")
    print(f" ➤ Features : {selected_feature_names}")

    print(" --- TRAIN (X_tr) ---")
    print(f"    R² TRAIN : {r2_tr:.4f} | MAE : {mae_tr:.4f} | RMSE : {rmse_tr:.4f}")

    print(" --- OOB (sur X_tr) ---")
    print(f"    R² OOB   : {oob_r2:.4f} | MAE : {oob_mae:.4f} | RMSE : {oob_rmse:.4f}")

    print(" --- VALIDATION (X_val) ---")
    print(f"    R² VAL   : {r2_val:.4f} | MAE : {mae_val:.4f} | RMSE : {rmse_val:.4f}")

    print(f" --- TEST (dates {int(TEST_FRACTION*100)}%) ---")
    print(f"    R² TEST  : {r2_test:.4f} | MAE : {mae_test:.4f} | RMSE : {rmse_test:.4f}")

    print(f" --- GAP (sur-apprentissage) ---")
    print(f"    GAP = R²_TRAIN - R²_VAL = {(r2_tr - r2_val):.4f}")
    print("╚══════════════════════════════════════════════════════╝")

    # ------------------------------------------------------------
    # 7.1) Ré-entrainement final sur tout X_train (cartographie + forecast)
    # ------------------------------------------------------------
    print("\n🔁 Ré-entrainement final sur TOUT X_train (cartographie + forecast)...")
    rf_model.fit(X_train, y_train)
    print("   ✔️ rf_model prêt.")

    # ------------------------------------------------------------
    # 8) CARTOGRAPHIE : prédictions sur les dates TEST (sauvegarde GeoTIFF)
    # ------------------------------------------------------------
    print(f"\n=== CARTOGRAPHIE : PREDICTIONS SUR LES DATES TEST ({int(TEST_FRACTION*100)}%) ===")

    def build_X_for_date_mapping(date_key, template_profile, role="map"):
        """Construit X pour TOUS les pixels de region_mask (pas besoin de y), retourne X_keep et idx_keep."""
        ndvi_date = get_ndvi_date_for(date_key)
        if ndvi_date is None:
            print(f"⏭️ [{role}] {date_key} : aucun NDVI <= date, ignoré.")
            return None, None

        et_date = get_et_date_for(date_key)
        if et_date is None:
            print(f"⏭️ [{role}] {date_key} : aucun ET <= date, ignoré.")
            return None, None

        if (
            date_key not in precip_dict
            or date_key not in api_dict
            or date_key not in lst_dict
            or date_key not in qsb_dict
            or date_key not in rzsm_dict
            or date_key not in soilm_dict
        ):
            print(f"⏭️ [{role}] {date_key} : variable journalière manquante, ignoré.")
            return None, None

        precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
        api_raw,    api_prof,    api_nodata    = read_raster_raw(api_dict[date_key])
        lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
        ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
        et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
        qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
        rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
        soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

        precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
        api_res    = reproject_to_template(api_raw,    api_prof,    template_profile, src_nodata=api_nodata,    dst_nodata=np.nan)
        lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata,    dst_nodata=np.nan)
        ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata,   dst_nodata=np.nan)
        et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata,     dst_nodata=np.nan)
        qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata,    dst_nodata=np.nan)
        rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata,   dst_nodata=np.nan)
        soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata,  dst_nodata=np.nan)

        update_region_mask(precip_res, lst_res, ndvi_res, template_profile)
        region_mask = get_region_mask(template_profile)

        precip_f = fill_nan_with_mean(precip_res)
        api_f    = fill_nan_with_mean(api_res)
        lst_f    = fill_nan_with_mean(lst_res)
        ndvi_f   = fill_nan_with_mean(ndvi_res)
        et_f     = fill_nan_with_mean(et_res)
        qsb_f    = fill_nan_with_mean(qsb_res)
        rzsm_f   = fill_nan_with_mean(rzsm_res)
        soilm_f  = fill_nan_with_mean(soilm_res)

        twi_2d  = ensure_twi_on_template(template_profile)
        dist_2d = ensure_dist_on_template(template_profile)

        mask_pred = region_mask
        if not np.any(mask_pred):
            print(f"⏭️ [{role}] {date_key} : region_mask vide, ignoré.")
            return None, None

        doy = date_key.timetuple().tm_yday
        sin_doy = np.sin(2 * np.pi * doy / 365.0)
        cos_doy = np.cos(2 * np.pi * doy / 365.0)

        n_pixels = np.count_nonzero(mask_pred)
        sin_plane = np.full(n_pixels, sin_doy, dtype="float32")
        cos_plane = np.full(n_pixels, cos_doy, dtype="float32")

        X_day = np.stack(
            [
                precip_f[mask_pred],
                api_f[mask_pred],
                lst_f[mask_pred],
                ndvi_f[mask_pred],
                et_f[mask_pred],
                qsb_f[mask_pred],
                rzsm_f[mask_pred],
                soilm_f[mask_pred],
                twi_2d[mask_pred],
                dist_2d[mask_pred],
                sin_plane,
                cos_plane,
            ],
            axis=1,
        )

        row_mask = (~np.isnan(X_day).any(axis=1))
        if not np.any(row_mask):
            print(f"⏭️ [{role}] {date_key} : pixels invalides (NaN features), ignoré.")
            return None, None

        mask_pred_flat = mask_pred.ravel()
        idx_pred = np.flatnonzero(mask_pred_flat)
        idx_keep = idx_pred[row_mask]

        X_keep = X_day[row_mask]

        # ✅ appliquer la même sélection que l'entraînement
        X_keep = X_keep[:, selected_idx]

        return X_keep, idx_keep

    for d in test_dates:
        X_map, idx_keep = build_X_for_date_mapping(d, template_profile, role="test-map")
        if X_map is None:
            continue

        pred = rf_model.predict(X_map).astype("float32")

        pred_full = np.full((template_profile["height"] * template_profile["width"],), np.nan, dtype="float32")
        pred_full[idx_keep] = pred
        pred_2d = pred_full.reshape((template_profile["height"], template_profile["width"]))

        out_path = os.path.join(OUT_DIR, f"GWS_RF_TESTPRED_{d.strftime('%Y%m%d')}.tif")
        write_geotiff(out_path, pred_2d, template_profile, nodata_value=NODATA_VALUE)
        print(f" ✅ [test-map] Sauvegardé : {out_path}")

    # ------------------------------------------------------------
    # 9) FORECAST : prédictions sur la période forecast (sauvegarde GeoTIFF)
    # ------------------------------------------------------------
    print(f"\n=== FORECAST : PREDICTIONS {FORECAST_START} → {FORECAST_END} ===")

    for d in date_range(FORECAST_START, FORECAST_END):
        if d in EXCLUDED_DATES:
            print(f"⏭️ [forecast] {d} : exclue (LST manquante).")
            continue

        X_map, idx_keep = build_X_for_date_mapping(d, template_profile, role="forecast")
        if X_map is None:
            continue

        pred = rf_model.predict(X_map).astype("float32")

        pred_full = np.full((template_profile["height"] * template_profile["width"],), np.nan, dtype="float32")
        pred_full[idx_keep] = pred
        pred_2d = pred_full.reshape((template_profile["height"], template_profile["width"]))

        out_path = os.path.join(OUT_DIR, f"GWS_RF_FORECAST_{d.strftime('%Y%m%d')}.tif")
        write_geotiff(out_path, pred_2d, template_profile, nodata_value=NODATA_VALUE)
        print(f" ✅ [forecast] Sauvegardé : {out_path}")

    print("\n🎉 Terminé : entraînement + OOB + régularisation + feature selection + test-map + forecast.")

## Exécution

Lance toute la chaîne : indexation → dataset → entraînement → évaluation → cartes prédictives.

In [ ]:
main()

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)